# Stage 3: Spatio-Temporal Analysis & Outputs
In this notebook, we focus on generating predictions, analyzing the changes over time, and extracting actionable insights (Decision Making) as described in the System Architecture diagram.
1. **Outputs (Predictions)**: Generating Water / Non-Water Classification masks using the trained Temporal Transformer Model.
2. **Spatio-Temporal Analysis**: Change Analysis ($t_n$ vs $t_{n-1}$) and Flood-Prone Assessment based on water frequency.
3. **Insights & Reports**: Visualizing the predicted masks over time.

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from scipy.ndimage import uniform_filter
import torch.nn as nn
from matplotlib.colors import ListedColormap

# Define the models again for loading weights
class SpatialEncoder(nn.Module):
    def __init__(self):
        super(SpatialEncoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(2, 16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(True),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.ReLU(True)
        )
    def forward(self, x):
        return self.encoder(x)

class TemporalTransformerPredictor(nn.Module):
    def __init__(self, seq_length=3, spatial_dim=128*16*16, embed_dim=512, num_heads=8, hidden_dim=512, num_layers=2):
        super(TemporalTransformerPredictor, self).__init__()
        self.spatial_encoder = SpatialEncoder()
        self.pre_proj = nn.Linear(spatial_dim, embed_dim)
        self.pos_embedding = nn.Parameter(torch.randn(1, seq_length, embed_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=hidden_dim, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.post_proj = nn.Linear(embed_dim, spatial_dim)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(True),
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(True),
            nn.ConvTranspose2d(16, 2, kernel_size=3, stride=2, padding=1, output_padding=1),
        )

    def forward(self, x):
        B, S, C, H, W = x.shape
        x_reshaped = x.view(B * S, C, H, W)
        spatial_features = self.spatial_encoder(x_reshaped)
        sf_flat = spatial_features.view(B, S, -1)
        sf_proj = self.pre_proj(sf_flat)
        sf_proj = sf_proj + self.pos_embedding
        temporal_features = self.transformer(sf_proj)
        last_step_features = temporal_features[:, -1, :]
        last_step_restored = self.post_proj(last_step_features)
        last_step_spatial = last_step_restored.view(B, 128, 16, 16)
        logits = self.decoder(last_step_spatial)
        return logits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TemporalTransformerPredictor(seq_length=3).to(device)

model_path = 'models/temporal_transformer.pth'
if os.path.exists(model_path):
    print("Loading trained temporal model...")
    model.load_state_dict(torch.load(model_path, weights_only=True))
    model.eval()
else:
    print("Model weights not found. Please train in Stage 2 first.")

### 1. Generating Predictions
We will load a test sequence and run it through the model to get a predicted water body mask.

In [ ]:
DATA_DIR = "data/sen1floods11"
S1_DIR = os.path.join(DATA_DIR, "S1Hand")

def apply_speckle_filter(image, size=3):
    filtered_image = np.zeros_like(image)
    for band in range(image.shape[0]):
        filtered_image[band] = uniform_filter(image[band], size=size)
    return filtered_image

def preprocess_s1(data):
    vv = np.nan_to_num(data[0], nan=-25.0)
    vh = np.nan_to_num(data[1], nan=-30.0)
    stacked = np.stack([vv, vh], axis=0)
    filtered = apply_speckle_filter(stacked, size=3)
    vv_clipped = np.clip(filtered[0], -30.0, 0.0)
    vh_clipped = np.clip(filtered[1], -35.0, -5.0)
    vv_norm = (vv_clipped - (-30.0)) / 30.0
    vh_norm = (vh_clipped - (-35.0)) / 30.0
    return np.stack([vv_norm, vh_norm], axis=0)

# Load a mock sequence of 3 images (using the downloaded test data)
s1_files = sorted([f for f in os.listdir(S1_DIR) if f.endswith('.tif')])
if len(s1_files) >= 3:
    seq_s1 = []
    for f in s1_files[-3:]: # Take the last 3 files
        with rasterio.open(os.path.join(S1_DIR, f)) as src:
            data = src.read()
            s1_preprocessed = preprocess_s1(data)
            s1_tensor = torch.tensor(s1_preprocessed, dtype=torch.float32).unsqueeze(0)
            s1_resized = torch.nn.functional.interpolate(
                s1_tensor, size=(256, 256), mode='bilinear', align_corners=False
            ).squeeze(0)
            seq_s1.append(s1_resized)
            
    seq_s1_tensor = torch.stack(seq_s1).unsqueeze(0).to(device) # Shape [1, 3, 2, 256, 256]
    
    with torch.no_grad():
        logits = model(seq_s1_tensor)
        predicted_mask = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy() # [256, 256]
        
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    # RGB composite of the last time step
    vv = seq_s1_tensor[0, -1, 0].cpu().numpy()
    vh = seq_s1_tensor[0, -1, 1].cpu().numpy()
    rgb = np.stack([vv, vh, vv-vh], axis=-1)
    rgb = (rgb - np.min(rgb)) / (np.max(rgb) - np.min(rgb) + 1e-8)
    plt.imshow(rgb)
    plt.title("Input SAR (t_n)")
    
    plt.subplot(1, 2, 2)
    cmap = ListedColormap(['gray', 'blue'])
    plt.imshow(predicted_mask, cmap=cmap)
    plt.title("Predicted Water Mask")
    plt.show()
else:
    print("Not enough images found.")

### 2. Spatio-Temporal Analysis
This section implements a basic Change Analysis ($t_n$ vs $t_{n-1}$) assuming we have predicted masks for both time steps. It highlights new flood areas.

In [ ]:
# Simulating an older predicted mask for demonstration
np.random.seed(42)
if len(s1_files) >= 3:
    # Let's say older mask had less water
    older_mask = np.copy(predicted_mask)
    older_mask[np.random.rand(256, 256) > 0.95] = 0 # Randomly remove some water
    
    # Change Detection Logic
    # 0 = No Water, 1 = Permanent Water, 2 = New Flood (Water at tn, not at tn-1), -1 = Receded
    change_map = np.zeros_like(predicted_mask, dtype=int)
    change_map[(older_mask == 1) & (predicted_mask == 1)] = 1
    change_map[(older_mask == 0) & (predicted_mask == 1)] = 2
    change_map[(older_mask == 1) & (predicted_mask == 0)] = -1
    
    plt.figure(figsize=(8, 8))
    cmap_change = ListedColormap(['lightgreen', 'lightgray', 'blue', 'red'])
    # lightgreen = receded, lightgray = no water, blue = permanent water, red = new flood
    bounds = [-1.5, -0.5, 0.5, 1.5, 2.5]
    norm = plt.cm.colors.BoundaryNorm(bounds, cmap_change.N)
    
    im = plt.imshow(change_map, cmap=cmap_change, norm=norm)
    cbar = plt.colorbar(im, ticks=[-1, 0, 1, 2])
    cbar.ax.set_yticklabels(['Receded', 'No Water', 'Permanent Water', 'New Flood'])
    plt.title("Temporal Water Change Detection (tn vs tn-1)")
    plt.show()
